In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce

In [2]:
df = pd.read_csv("../data/dataset_mood_smartphone.csv", index_col=0)
df["time"] = pd.to_datetime(df["time"])


In [3]:
#kunnen we later ook nog makkelijk aanpassen naar max ofzo 
# Bijvoorbeeld voor "activity", "circumplex.arousal", "circumplex.valence"
def get_agg(var):
    if var in ["mood", "activity", "circumplex.arousal", "circumplex.valence"]:
        return "mean"
    
    elif "appCat" in var:
        return "sum"
    
    elif var in ["screen","sms", "call"]:
        return "sum"
    
    else:
        return "mean"  # fallback

In [ ]:
# hoeveel uur/dagen per keer
freq = "D"  #of "h", "3h", "6h", "D" etc.

user_datasets = {}

for user_id in df["id"].unique():
    
    df_user = df[df["id"] == user_id].copy()
    df_user["time"] = pd.to_datetime(df_user["time"])
    
    # -----------------------------
    # mood-based range
    # -----------------------------
    df_mood = df_user[df_user["variable"] == "mood"]
    
    if df_mood.empty:
        print(f"{user_id}: geen mood data -> skip")
        continue
    
    start = df_mood["time"].min().floor("D")
    end = df_mood["time"].max().ceil("D")
    
    full_index = pd.date_range(start=start, end=end, freq=freq)
    
    # -----------------------------
    # pivot
    # -----------------------------
    df_pivot = df_user.pivot_table(
        index="time",
        columns="variable",
        values="value",
        aggfunc="mean"
    ).sort_index()
    
    # -----------------------------
    # resample per variabele
    # -----------------------------
    df_resampled = pd.DataFrame(index=full_index)

    for col in df_pivot.columns:
        agg = get_agg(col)
        
        if agg == "mean":
            df_resampled[col] = df_pivot[col].resample(freq).mean()
        
        elif agg == "sum":
            df_resampled[col] = df_pivot[col].resample(freq).sum().fillna(0) #geen waarde, dus geen occurence dus 0


    # index -> kolom
    df_resampled = df_resampled.reset_index().rename(columns={"index": "time_bin"})
    
    # -----------------------------
    # opslaan per user
    # -----------------------------
    user_datasets[user_id] = df_resampled

Het is dus een dictonary met user ID als namen en het df als item, voorbeeld:

In [5]:
user_datasets['AS14.06']

,time_bin,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.game,appCat.office,appCat.other,appCat.social,appCat.travel,appCat.unknown,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,2014-03-24,0.043432,43.495,1100.514,1073.377,0.000,0.000,220.348,292.297,0.000,0.000,0.000,3.0,-1.000000,0.000000,7.000000,2508.255000,0.0
1,2014-03-25,0.137709,116.887,3930.897,2999.969,0.000,276.938,112.220,874.732,0.000,0.000,0.000,0.0,-0.200000,0.600000,7.000000,7729.770001,0.0
2,2014-03-26,0.135563,212.512,3234.500,1950.957,0.000,35.692,60.047,3143.653,564.176,0.000,0.000,2.0,0.500000,1.500000,8.000000,7892.531001,0.0
3,2014-03-27,0.132329,523.696,4045.840,2453.259,0.000,0.000,55.648,1081.538,0.000,0.000,0.000,1.0,0.250000,0.250000,7.000000,8167.157001,0.0
4,2014-03-28,0.171471,42.311,3588.191,1995.996,0.000,42.263,367.836,3417.103,80.015,0.000,0.000,0.0,-0.200000,0.800000,7.600000,6928.638002,1.0
5,2014-03-29,0.111581,32.411,1881.483,786.555,0.000,0.000,101.467,1257.523,0.000,0.000,0.000,0.0,0.000000,1.000000,7.800000,3907.131000,0.0
6,2014-03-30,0.211298,1874.664,549.846,2057.889,2386.332,0.000,44.391,2072.094,0.000,0.000,4.859,1.0,0.000000,0.800000,7.800000,8933.720999,4.0
7,2014-03-31,0.160605,2017.195,2061.095,2416.332,0.000,27.522,49.306,2699.895,0.000,0.000,0.000,1.0,-0.200000,0.400000,7.000000,7504.322001,0.0
8,2014-04-01,0.225854,1351.484,4034.721,775.560,866.915,0.000,56.761,2224.119,0.000,0.000,0.000,3.0,0.000000,0.750000,7.750000,8370.870000,0.0
9,2014-04-02,0.162553,597.655,4435.702,1418.497,0.000,167.000,58.694,2845.707,22.122,0.000,0.000,4.0,-0.600000,0.400000,6.800000,9183.504001,0.0
